# Sentinel-2 scenes of the Shisper lake

Exports Sentinel-2 Level-2A surface reflectance for 74.50 to 74.62° E and 36.30 to 36.40° N on the acquisition dates of the GLOFNet archive (`data/s2_dates.txt`): bands B4, B3, B2, B8, B11 and B12 at 10 m in UTM zone 43N, from the Microsoft Planetary Computer (no account needed). The reflectance offset of processing baseline 04.00 (scenes from 25 January 2022) is removed at export, so all years share one scale.

The notebook also maps the lake as open water on 31 May 2019 that is absent on 4 June 2023 and saves its centre to `lake_location.json`; RiskFlow centres its 5 km window on this point.

**Output** (`Shisper_lake_S2`): one `YYYY-MM-DD.tif` per date, `lake_location.json`, `export_info.json` and `lake_check.png`. CPU runtime, about 30 minutes; finished dates are skipped on a rerun.

## Setup

In [ ]:
%pip install -q pystac-client planetary-computer odc-stac rasterio

import os, json, time
import numpy as np, pandas as pd
import pystac_client, planetary_computer, odc.stac, odc.geo.xr, rasterio
from rasterio.warp import transform as warp_transform
from scipy import ndimage
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

DRIVE = '/content/drive/MyDrive'
OUT = f'{DRIVE}/Shisper_lake_S2'
REPO = 'https://raw.githubusercontent.com/ZuhaFatima96/IceWatch-A-Multimodal-Deep-Learning-Framework-for-Early-Warning-of-GLOFs/main'
DATA = '../data' if os.path.isdir('../data') else f'{REPO}/data'
os.makedirs(OUT, exist_ok=True)

AOI = (74.50, 36.30, 74.62, 36.40)                  # west, south, east, north
BANDS = ['B04', 'B03', 'B02', 'B08', 'B11', 'B12']
NAMES = ('B4', 'B3', 'B2', 'B8', 'B11', 'B12')
CRS, RES = 'EPSG:32643', 10

DATES = sorted(pd.read_csv(f'{DATA}/s2_dates.txt', header=None)[0].astype(str))
print(len(DATES), 'dates,', DATES[0], 'to', DATES[-1])
json.dump(dict(source='Microsoft Planetary Computer, sentinel-2-l2a', aoi_west_south_east_north=AOI, crs=CRS,
               resolution_m=RES, bands=NAMES, harmonised_offset=True, dates='data/s2_dates.txt'),
          open(f'{OUT}/export_info.json', 'w'), indent=2)

odc.stac.configure_rio(cloud_defaults=True)
catalog = pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1',
                                    modifier=planetary_computer.sign_inplace)

## Lake check

In [ ]:
def load_day(d):
    items = catalog.search(collections=['sentinel-2-l2a'], bbox=AOI, datetime=d).item_collection()
    if len(items) == 0:
        return None, None
    ds = odc.stac.load(items, bands=BANDS, bbox=AOI, crs=CRS, resolution=RES, groupby='solar_day',
                       resampling='nearest', stac_cfg={'*': {'assets': {'*': {'data_type': 'uint16', 'nodata': 0}}}})
    arr = np.stack([ds[b].isel(time=0).values for b in BANDS]).astype('uint16')
    vals = []
    for it in items:
        try:
            vals.append(float(it.properties.get('s2:processing_baseline')))
        except (TypeError, ValueError):
            pass
    shifted = max(vals) >= 4.0 if vals else d >= '2022-01-25'
    if shifted:                     # processing baseline 04.00 and later: remove the +1000 offset
        arr = np.where(arr > 0, np.clip(arr.astype('int32') - 1000, 1, None), 0).astype('uint16')
    return arr, ds.odc.geobox

def save_tif(arr, gbox, path):
    prof = dict(driver='GTiff', height=arr.shape[1], width=arr.shape[2], count=arr.shape[0], dtype='uint16',
                crs=f'EPSG:{gbox.crs.epsg}', transform=gbox.affine, nodata=0, compress='deflate', tiled=True)
    with rasterio.open(path, 'w', **prof) as dst:
        dst.write(arr)
        dst.descriptions = NAMES

def water(arr):
    g, nir, sw = (arr[i].astype('float32') for i in (1, 3, 4))   # B3, B8, B11
    ndwi = (g - nir) / (g + nir + 1e-6)
    return (ndwi > 0.15) & (nir < 1500) & (sw < 1000) & (g > 0)

def rgb(arr):
    x = np.transpose(arr[[0, 1, 2]], (1, 2, 0)).astype('float32')
    v = x[x > 0]
    lo, hi = (np.percentile(v, 2), np.percentile(v, 98)) if v.size else (0, 1)
    return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1)

CHECK = [('2019-05-31', 'lake full'), ('2020-05-25', 'lake full, 4 days before the outburst'), ('2023-06-04', 'no lake')]
loaded = {}
for d, _ in CHECK:
    if d in DATES:
        arr, gbox = load_day(d)
        if arr is not None:
            loaded[d] = (arr, gbox)
            save_tif(arr, gbox, f'{OUT}/{d}.tif')

lake = None
if '2019-05-31' in loaded:
    cand = water(loaded['2019-05-31'][0])
    if '2023-06-04' in loaded:
        cand &= ~ndimage.binary_dilation(water(loaded['2023-06-04'][0]), iterations=2)
    if '2020-05-25' in loaded:
        both = cand & ndimage.binary_dilation(water(loaded['2020-05-25'][0]), iterations=5)
        if both.sum() > 300:
            cand = both
    lab, n = ndimage.label(cand)
    if n:
        sizes = ndimage.sum(cand, lab, range(1, n + 1))
        k = int(np.argmax(sizes)) + 1
        rr, cc = np.nonzero(lab == k)
        gbox = loaded['2019-05-31'][1]
        x, y = gbox.affine * (cc.mean() + 0.5, rr.mean() + 0.5)
        lon, lat = warp_transform(f'EPSG:{gbox.crs.epsg}', 'EPSG:4326', [x], [y])
        lake = dict(lat=round(float(lat[0]), 5), lon=round(float(lon[0]), 5),
                    area_km2=round(float(sizes[k - 1]) * RES * RES / 1e6, 3), found_on='2019-05-31',
                    method='water on 31 May 2019 (and 25 May 2020), absent on 4 Jun 2023; largest patch')
        LAKE_MASK = lab == k
print('lake:', lake)

fig, ax = plt.subplots(1, len(loaded), figsize=(6 * max(len(loaded), 1), 6), squeeze=False)
for j, (d, (arr, gbox)) in enumerate(loaded.items()):
    ax[0, j].imshow(rgb(arr))
    if lake is not None:
        ax[0, j].contour(LAKE_MASK, levels=[0.5], colors='red', linewidths=1.2)
    ax[0, j].set_title(f'{d}: ' + dict(CHECK)[d], fontsize=11)
    ax[0, j].set_xticks([]); ax[0, j].set_yticks([])
plt.suptitle('Red outline: detected lake', fontsize=12)
plt.tight_layout(); plt.savefig(f'{OUT}/lake_check.png', dpi=110); plt.show()
if lake is not None and lake['area_km2'] >= 0.05:
    json.dump(lake, open(f'{OUT}/lake_location.json', 'w'), indent=2)
else:
    print('No lake found: check lake_check.png before the export.')

## Export

In [ ]:
missing, t0 = [], time.time()
for i, d in enumerate(DATES):
    out = f'{OUT}/{d}.tif'
    if os.path.exists(out):
        continue
    arr, err = None, None
    for k in range(3):
        try:
            arr, gbox = load_day(d)
            break
        except Exception as e:
            err = e
            time.sleep(5 * (k + 1))
    if arr is None:
        missing.append(d)
        print(f'  no data for {d}', f'({str(err)[:120]})' if err else '(no scene in the catalogue)', flush=True)
        continue
    save_tif(arr, gbox, out)
    if i % 10 == 0:
        print(f'  {i + 1}/{len(DATES)}  {d}  {arr.shape[1]} x {arr.shape[2]} px  ({(time.time() - t0) / 60:.1f} min)', flush=True)
n = sum(os.path.exists(f'{OUT}/{d}.tif') for d in DATES)
print(f'{n} of {len(DATES)} dates exported | missing:', ', '.join(missing) if missing else 'none')